# Descargar multiples genomas del NCBI

Desde Conda, isntalaremos un environment para poder descargar las secuencias de NCBI
```Bash:
conda create -n ncbi_env python=3.8 -y
conda activate ncbi_env
```

Instalar la dependencia de NCBI
```Bash:
conda install -c bioconda ncbi-genome-download ncbi-datasets-cli -y
```

```Bash:
pip install ncbi-genome-download
```
Este como opcion en caso de que Conda no lo haga

### Environment Check

Con el Environment hecho, solo necesitamos una lista con los IDs de los genomas, se puede hacer en un archivo .csv, pero que no tenga encabezados, solo los ID de lso genomas que queremos 

### Descargar genomas
Debemos crear un directorio donde solo este el .csv de los ID de los genomas y el script que correra Bash
```Bash:
mkdir descargar_genomas
cd descargar_genomas
```

En el directorio de "descargar_genomas", creamos un archivo .sh,
esto se puede hacer colocando:
```Bash:
nano
```

```Bash:
#!/bin/bash

# metadata
metadata="candidatos_genomas_cladoA.csv"

Red="$(tput setaf 1)"
Green="$(tput setaf 2)"
Bold=$(tput bold)
reset=$(tput sgr0)

while IFS=, read -r field1; do
    # Omitir líneas vacías o espacios en blanco
    [[ -z "$field1" ]] && continue

    # Limpiar posibles retornos de carro (\r) si el CSV se hizo en Windows
    field1=$(echo "$field1" | tr -d '\r' | xargs)

    echo ""
    echo "${Red}${Bold}Downloading ${reset}: ${field1}"

    # 1. Limpiar restos de ejecuciones anteriores
    rm -rf "${field1}.zip" ncbi_dataset* README.md md5sum.txt

    # 2. Descargar
    datasets download genome accession "${field1}" --filename "${field1}.zip"

    if [ ! -f "${field1}.zip" ]; then
        echo "${Red}Error: No se pudo descargar ${field1}. Pasando al siguiente...${reset}"
        continue
    fi

    echo "${Bold}Extracting ${field1}.zip ${reset}"
    unzip -q "${field1}.zip"

    # 3. Entrar a la carpeta de manera segura con subshell (paréntesis)
    (
        cd "ncbi_dataset/data/${field1}" || exit 1
        echo "${Bold}Moving ${field1} fasta file into home directory${reset}"
        mv *.fna ../../../
    )

done < "$metadata"
```

Con estos dos archivos, solo hace fala tener activado en environment de ncbi y correr el archivo .sh
```Bash:
./descargar_genomas.sh
```